<a href="https://colab.research.google.com/github/JasmineLee7/movie-rec/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Milestone 2!")

Milestone 2


#Part A: Load the MovieLens ratings

In [7]:
import os, urllib.request, zipfile, ssl
import numpy as np
import pandas as pd

if not os.path.exists("ml-100k"):
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    with urllib.request.urlopen(
            "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
            context=ctx) as resp, open("ml-100k.zip", "wb") as f:
        f.write(resp.read())
    zipfile.ZipFile("ml-100k.zip").extractall()

ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "item", "rating", "time"])

print(ratings.head())
print("Number of ratings:", len(ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


This code loads the MovieLens 100K ratings. It first imports the tools for downloading files, unzipping them, and working with data (NumPy and Pandas). If the `ml-100k` folder doesn't already exist, it downloads and unzips the dataset, so rerunning the notebook doesn't download it again. Then `pd.read_csv` reads the tab-separated ratings file into a table with four columns: user, item (movie), rating, and time. Each row is one known rating, so the data is in long form. The printout shows the first few rows and confirms there are 100,000 ratings.

# Part B: Make and honest train/test split

In [12]:
def split_ratings(ratings, test_frac=0.10, seed=0):
  # Hide ratings without removing a user's or movie's last training rating.
  rng = np.random.default_rng(seed)
  is_test = np.zeros(len(ratings), dtype=bool)

  user_left = ratings.groupby("user").size().to_dict()
  item_left = ratings.groupby("item").size().to_dict()

  for pos in rng.permutation(len(ratings)):
      row = ratings.iloc[pos]
      user, item = row["user"], row["item"]

      if (rng.random() < test_frac
            and user_left[user] > 1
            and item_left[item] > 1):
        is_test[pos] = True
        user_left[user] -= 1
        item_left[item] -= 1

  return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

train, test = split_ratings(ratings)
print("train:", len(train), "test:", len(test))
print("test fraction:", round(len(test) / len(ratings), 3))

train: 90032 test: 9968
test fraction: 0.1


This function splits the ratings into training and test sets so the recommendation system can be fairly evaluated. It visits the ratings in a random order, giving each one around a 10% of being place in the test set. It returns train, which are the ratings used to build the recommendation models, and test, which are hidden ratings used to check prediction accuracy.

# Part C: Build the training matrix

In [ ]:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
    """Return a user-by-movie matrix; missing ratings are np.nan."""
    R = np.full((N_USERS, N_ITEMS), np.nan)
    R[df["user"].to_numpy() - 1,
      df["item"].to_numpy() - 1] = df["rating"].to_numpy()
    return R

R = to_matrix(train)
known = ~np.isnan(R)

print("Matrix shape:", R.shape)
print("Known training ratings:", known.sum())

This code turns the training table into a matrix with 943 user rows and 1,682 movie columns. It starts with every entry set to `np.nan`, meaning unknown, and fills in only the ratings in `train`. We subtract 1 from user and movie IDs because the IDs start at 1 but Python positions start at 0. The Boolean array `known` is True wherever the matrix contains a training rating. Its sum should equal the number of rows in `train`. Test ratings stay missing in this matrix so the models cannot learn from the answers they will be tested on.

In [ ]:
truth = test["rating"].to_numpy()
u = test["user"].to_numpy() - 1
i = test["item"].to_numpy() - 1

This code prepares the hidden ratings for evaluation. `truth` stores the actual test ratings, while `u` and `i` store their user row and movie column positions, adjusted to start at 0. Matching positions in these three arrays refer to the same test example. Later sections will use `u` and `i` to look up predictions and compare them with `truth`. These test ratings are used only for scoring; they are not added to the training matrix.